### Structured output
Models can be requested to provide their response in a format matching a given schema. This is useful for
ensuring the output can be easily parsed and used in subsequent processing. LangChain supports multiple
schema types and methods for enforcing structured output.

### Pydantic
Pydantic models provide the richest feature set with field validation, descriptions, and nested structures-

In [1]:
import os 
from langchain.chat_models import init_chat_model
os.environ["GROQ_API_KEY"] = os.getenv("GROQ_API_KEY")
model=init_chat_model("groq:qwen/qwen3-32b")
model

ChatGroq(metadata={'lc_versions': {'langchain-core': '1.4.7', 'langchain': '1.3.9'}}, output_version=None, profile={'name': 'Qwen3 32B', 'release_date': '2024-12-23', 'last_updated': '2024-12-23', 'open_weights': True, 'max_input_tokens': 131072, 'max_output_tokens': 40960, 'text_inputs': True, 'image_inputs': False, 'audio_inputs': False, 'video_inputs': False, 'text_outputs': True, 'image_outputs': False, 'audio_outputs': False, 'video_outputs': False, 'reasoning_output': True, 'tool_calling': True, 'attachment': False, 'temperature': True}, client=<groq.resources.chat.completions.Completions object at 0x000002370F114F50>, async_client=<groq.resources.chat.completions.AsyncCompletions object at 0x0000023710241510>, model_name='qwen/qwen3-32b', model_kwargs={}, groq_api_key=SecretStr('**********'), groq_api_base=None, groq_proxy=None)

In [2]:
from pydantic import BaseModel,Field

class Movie(BaseModel):
    title:str=Field(description="Name of the movie")
    year:int=Field(description="This year the movie was released")
    director:str=Field(description="The director of the movie")
    rating:float=Field(description="The movie's rating out of 10")

In [3]:
model_with_structure=model.with_structured_output(Movie)
model_with_structure

_ChatModelBinding(bound=ChatGroq(metadata={'lc_versions': {'langchain-core': '1.4.7', 'langchain': '1.3.9'}}, output_version=None, profile={'name': 'Qwen3 32B', 'release_date': '2024-12-23', 'last_updated': '2024-12-23', 'open_weights': True, 'max_input_tokens': 131072, 'max_output_tokens': 40960, 'text_inputs': True, 'image_inputs': False, 'audio_inputs': False, 'video_inputs': False, 'text_outputs': True, 'image_outputs': False, 'audio_outputs': False, 'video_outputs': False, 'reasoning_output': True, 'tool_calling': True, 'attachment': False, 'temperature': True}, client=<groq.resources.chat.completions.Completions object at 0x000002370F114F50>, async_client=<groq.resources.chat.completions.AsyncCompletions object at 0x0000023710241510>, model_name='qwen/qwen3-32b', model_kwargs={}, groq_api_key=SecretStr('**********'), groq_api_base=None, groq_proxy=None), kwargs={'tools': [{'type': 'function', 'function': {'name': 'Movie', 'description': '', 'parameters': {'properties': {'title': 

In [5]:
# Regular output by the model
model.invoke("Provide details about the movie Transformers")

AIMessage(content='<think>\nOkay, so I need to provide details about the movie Transformers. Let me start by recalling what I know about it. Transformers is a movie that came out in 2007, right? It\'s based on the Transformers toy line and the animated series from the 80s. The director is Michael Bay, who\'s known for making action-packed films like The Transformers. The main characters are the Autobots and Decepticons, which are alien robots that can transform into vehicles and machines. \n\nThe story is probably set in the present day, but I think there\'s also a part set in the past, like the Cold War era. The main human characters might include a teenager who discovers a Transformer, and a military figure. Optimus Prime is the leader of the Autobots, and Megatron leads the Decepticons. The Autobots are trying to find a powerful artifact, maybe the AllSpark, which can create life. The Decepticons are after it too, leading to a conflict on Earth.\n\nI should mention the main cast. Sh

In [6]:
# Output after using pydantic
model_with_structure.invoke("Provide details about the movie Transformers")

Movie(title='Transformers', year=2007, director='Michael Bay', rating=6.8)

### Nested Structure

In [9]:
from pydantic import BaseModel, Field

class Actor(BaseModel):
    name: str
    role: str

class MovieDetails(BaseModel):
    title: str
    year: int
    cast: list[Actor]
    genres: list[str]
    budget: float | None = Field(None, description="Budget in millions USD")

model_with_structure = model.with_structured_output(MovieDetails)

response = model_with_structure.invoke("Provide details about the movie transformers")
response


MovieDetails(title='Transformers', year=2007, cast=[Actor(name='Shia LaBeouf', role='Sam Witwicky'), Actor(name='Megan Fox', role='Mikaela Banes'), Actor(name='Josh Duhamel', role='Bumblebee'), Actor(name='Tyrese Gibson', role='Gospel')], genres=['Action', 'Science Fiction'], budget=150.0)

### TypedDict
TypedDict provides a simpler alternative using Python's built-in typing, ideal when you dont need runtime validation.

In [11]:
from typing_extensions import TypedDict, Annotated

class MovieDict(TypedDict):
    """A movie with details"""
    title:Annotated[str,...,"Name of the movie"]
    year:Annotated[int,...,"This year the movie was released"]
    director:Annotated[str,...,"The director of the movie"]
    rating:Annotated[float,...,"The movie's rating out of 10"]

model_withtypedict = model.with_structured_output(MovieDict)
response = model_withtypedict.invoke("Please provide the details of the movie Avengers")
response

{'director': 'Joss Whedon', 'rating': 8, 'title': 'The Avengers', 'year': 2012}

In [13]:
class Actor(TypedDict):
    name: str
    role: str

class MovieDetails(TypedDict):
    title: str
    year: int
    cast: list[Actor]
    genres: list[str]
    budget: float | None = Field(None, description="Budget in millions USD")

model_with_structure = model.with_structured_output(MovieDetails)

response = model_with_structure.invoke("Provide details about the movie Avengers")
response

{'budget': 220500000,
 'cast': [{'name': 'Robert Downey Jr.', 'role': 'Tony Stark / Iron Man'},
  {'name': 'Chris Evans', 'role': 'Steve Rogers / Captain America'},
  {'name': 'Mark Ruffalo', 'role': 'Bruce Banner / Hulk'},
  {'name': 'Chris Hemsworth', 'role': 'Thor'},
  {'name': 'Scarlett Johansson', 'role': 'Natasha Romanoff / Black Widow'},
  {'name': 'Jeremy Renner', 'role': 'Clint Barton / Hawkeye'},
  {'name': 'Samuel L. Jackson', 'role': 'Nick Fury'}],
 'genres': ['Action', 'Adventure', 'Sci-Fi', 'Superhero'],
 'title': 'Avengers',
 'year': 2012}